In [2]:
# import libraries
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast, DistilBertModel
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from my_dataset import ToxicCommentDataset
from sklearn.metrics import precision_score, recall_score, f1_score

In [2]:
# import dataset
df = pd.read_csv('data.csv')
df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159571 entries, 0 to 159570
Data columns (total 8 columns):
 #   Column         Non-Null Count   Dtype 
---  ------         --------------   ----- 
 0   id             159571 non-null  object
 1   comment_text   159571 non-null  object
 2   toxic          159571 non-null  int64 
 3   severe_toxic   159571 non-null  int64 
 4   obscene        159571 non-null  int64 
 5   threat         159571 non-null  int64 
 6   insult         159571 non-null  int64 
 7   identity_hate  159571 non-null  int64 
dtypes: int64(6), object(2)
memory usage: 9.7+ MB


In [4]:
# data analysis
print("Missing value check:")
print(df.isnull().sum().sum())

print("\nTarget classes distributions:")
target_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
print(df[target_cols].sum())

print("\nClean comments:")
clean_comments = (df[target_cols].sum(axis=1) == 0).sum()
print(f"{clean_comments} ({clean_comments / len(df) * 100:.2f}%)")


Missing value check:
0

Target classes distributions:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64

Clean comments:
143346 (89.83%)


In [5]:
# data cleaning 
def clean_text(text):
    if not isinstance(text, str):
        return ""
    
    # new line or tab character to space
    text = text.replace("\n", " ").replace("\t", " ")

    # clean URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # clean IP addresses 
    text = re.sub(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}', '', text)

    # remove unnecessary spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [6]:
# apply cleaning to comment text
df['cleaned_text'] = df['comment_text'].apply(clean_text)

# calculate word counts
df['word_count'] = df['cleaned_text'].apply(lambda x: len(x.split()))

print("Word counts: ")
print(df['word_count'].describe())

print("\nPercentiles: ")
print("Word count 50% of comments:", df['word_count'].quantile(0.50))
print("Word count 75% of comments:", df['word_count'].quantile(0.75))
print("Word count 90% of comments:", df['word_count'].quantile(0.90))
print("Word count 95% of comments:", df['word_count'].quantile(0.95))
print("Word count 98% of comments:", df['word_count'].quantile(0.98))

Word counts: 
count    159571.000000
mean         67.178322
std          99.174848
min           0.000000
25%          17.000000
50%          36.000000
75%          75.000000
max        1411.000000
Name: word_count, dtype: float64

Percentiles: 
Word count 50% of comments: 36.0
Word count 75% of comments: 75.0
Word count 90% of comments: 152.0
Word count 95% of comments: 229.0
Word count 98% of comments: 386.0


In [7]:
# remove rows with more than 256 word counts (only 4% of data)
df_filtered = df[df['word_count']<=256].copy()

In [8]:
# producing subsets
toxic_df = df_filtered[df_filtered[target_cols].sum(axis=1) > 0].copy()

clean_df = df_filtered[df_filtered[target_cols].sum(axis=1)== 0].copy()

# downsample clean df to prevent class unbalance
toxic_downsample = toxic_df.sample(3000, random_state=42)
clean_downsample = clean_df.sample(5000, random_state=42)

final_df = pd.concat([toxic_downsample, clean_downsample]).sample(frac=1, random_state=42).reset_index(drop=True)
print("New Version:")
print(f"Shape: {final_df.shape}")
new_clean_ratio = (final_df[target_cols].sum(axis=1) == 0).sum() / len(final_df) * 100
print(f"Clean comment ratio: %{new_clean_ratio:.2f}")
print(f"Toxic comment ratio: %{100 - new_clean_ratio:.2f}")

New Version:
Shape: (8000, 10)
Clean comment ratio: %62.50
Toxic comment ratio: %37.50


In [9]:
train_df, test_df = train_test_split(final_df, test_size=0.2, random_state=42)

In [10]:
# save test set as csv 
test_df.to_csv('clean_test_set.csv', index=False)
train_df.to_csv('clean_train_set.csv', index=False)

In [4]:
class DistilBertMultiLabelClassifier(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased', num_labels=6):
        super(DistilBertMultiLabelClassifier, self).__init__()
        # Load pre-trained DistilBERT model
        self.distilbert = DistilBertModel.from_pretrained(model_name)
        # Dropout layer to prevent overfitting
        self.dropout = nn.Dropout(0.2)
        # Classification head: a linear layer that maps the [CLS] token representation to the number of labels
        self.classifier = nn.Linear(self.distilbert.config.hidden_size, num_labels)
        
    def forward(self, input_ids, attention_mask):
        # Pass inputs through DistilBERT
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Get the hidden state of the last layer (the output of DistilBERT)
        hidden_state = outputs[0]
        
        # Use the hidden state of the [CLS] token (the first token) for classification
        cls_output = hidden_state[:, 0, :] # [batch_size, 768]
        
        cls_output = self.dropout(cls_output)
        logits = self.classifier(cls_output)
        return logits

In [12]:
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np

# train_df'imizin indekslerini sıfırlıyoruz
train_df = train_df.reset_index(drop=True)
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# 1. Fold sütununu sıfırla
train_df['fold'] = -1

# 2. ÇOKLU ETİKET DAĞILIMINI KORUYAN AKILLI STRATIFICATION ALGORİTMASI
# Her bir etiket için veriyi kendi içinde fold'lara dengeli dağıtıyoruz
for label_idx, col in enumerate(target_cols):
    # Sadece bu etiket için pozitif olan satırları bul
    pos_indices = train_df[train_df[col] == 1].index.values
    
    # Bu pozitif satırları 5 fold'a eşit böl
    kf = KFold(n_splits=5, shuffle=True, random_state=42 + label_idx)
    for fold_id, (_, val_idx) in enumerate(kf.split(pos_indices)):
        # Henüz fold atanmamış olanlara ata
        actual_val_indices = pos_indices[val_idx]
        train_df.loc[actual_val_indices, 'fold'] = fold_id

# 3. Geriye kalan tamamen temiz (0-0-0-0-0-0) verileri de 5 fold'a eşit dağıtıyoruz
clean_indices = train_df[train_df[target_cols].sum(axis=1) == 0].index.values
kf_clean = KFold(n_splits=5, shuffle=True, random_state=99)
for fold_id, (_, val_idx) in enumerate(kf_clean.split(clean_indices)):
    actual_clean_indices = clean_indices[val_idx]
    train_df.loc[actual_clean_indices, 'fold'] = fold_id

# --- KONTROL RAPORU ---
print("--- New Multi-Label Fold Distributions (Should be almost equal) ---")
for fold_id in range(5):
    fold_sum = train_df[train_df['fold'] == fold_id][target_cols].sum()
    print(f"\nFold {fold_id} Target Counts:")
    print(fold_sum)

--- New Multi-Label Fold Distributions (Should be almost equal) ---

Fold 0 Target Counts:
toxic            473
severe_toxic      31
obscene          257
threat            12
insult           235
identity_hate     40
dtype: int64

Fold 1 Target Counts:
toxic            455
severe_toxic      43
obscene          266
threat            14
insult           243
identity_hate     39
dtype: int64

Fold 2 Target Counts:
toxic            443
severe_toxic      44
obscene          249
threat            15
insult           240
identity_hate     39
dtype: int64

Fold 3 Target Counts:
toxic            442
severe_toxic      54
obscene          254
threat            12
insult           225
identity_hate     39
dtype: int64

Fold 4 Target Counts:
toxic            439
severe_toxic      40
obscene          249
threat            10
insult           234
identity_hate     39
dtype: int64


In [13]:
def train_and_evaluate_fold(fold_id, train_df, tokenizer, device, epochs=1, batch_size=64):
    print(f"\n--- FOLD {fold_id} TRAINING ---")
    
    # Target label list for classification metrics
    target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    
    # 1. Split train and validation folds from train_df
    fold_train_data = train_df[train_df['fold'] != fold_id].reset_index(drop=True)
    fold_val_data = train_df[train_df['fold'] == fold_id].reset_index(drop=True)
    
    # Create Dataset instances
    train_dataset = ToxicCommentDataset(fold_train_data, tokenizer, max_len=256)
    val_dataset = ToxicCommentDataset(fold_val_data, tokenizer, max_len=256)
    
    # DataLoaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=False)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=False)
    
    # 2. Initialize Model, Loss Function, and Optimizer
    model = DistilBertMultiLabelClassifier().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    
    total_steps = len(train_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    
    best_val_loss = float('inf')
    best_macro_f1 = 0
    
    # 3. Epoch Loop (Set to 1 Epoch)
    for epoch in range(epochs):
        print(f"\n--- Epoch {epoch+1}/{epochs} ---")
        model.train()
        train_loss = 0
        
        for step, batch in enumerate(train_loader):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            
            train_loss += loss.item()
            
            # Print batch status every 20 steps
            if step % 20 == 0:
                print(f"Batch {step}/{len(train_loader)} | Batch Loss: {loss.item():.4f}")
            
        avg_train_loss = train_loss / len(train_loader)
        
        # 4. Validation Loop
        model.eval()
        val_loss = 0
        all_labels = []
        all_preds = []
        
        print("\nEvaluating on validation set...")
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                val_loss += loss.item()
                
                probs = torch.sigmoid(logits).cpu().numpy()
                all_labels.append(labels.cpu().numpy())
                all_preds.append(probs)
                
        avg_val_loss = val_loss / len(val_loader)
        all_labels = np.vstack(all_labels)
        all_preds = np.vstack(all_preds)
        
        # Binarize predictions based on threshold = 0.5
        binary_preds = (all_preds >= 0.5).astype(int)
        
        # --- METRIC CALCULATIONS ---
        
        # A) Per-Class Precision, Recall, and F1-Score
        class_precisions = precision_score(all_labels, binary_preds, average=None, zero_division=0)
        class_recalls = recall_score(all_labels, binary_preds, average=None, zero_division=0)
        class_f1s = f1_score(all_labels, binary_preds, average=None, zero_division=0)
        
        # B) Global Averages (Macro & Micro)
        macro_precision = precision_score(all_labels, binary_preds, average='macro', zero_division=0)
        macro_recall = recall_score(all_labels, binary_preds, average='macro', zero_division=0)
        macro_f1 = f1_score(all_labels, binary_preds, average='macro', zero_division=0)
        
        micro_precision = precision_score(all_labels, binary_preds, average='micro', zero_division=0)
        micro_recall = recall_score(all_labels, binary_preds, average='micro', zero_division=0)
        micro_micro_f1 = f1_score(all_labels, binary_preds, average='micro', zero_division=0)
        
        # --- PRINT PERFORMANCE REPORT ---
        print(f"\n--- Epoch {epoch+1} Summary Report ---")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        
        print("\nPer-Class Performance:")
        print(f"{'Label':<15} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10}")
        print("-" * 55)
        for idx, col_name in enumerate(target_cols):
            print(f"{col_name:<15} | {class_precisions[idx]:.4f}     | {class_recalls[idx]:.4f}   | {class_f1s[idx]:.4f}")
            
        print("\nGlobal Averages:")
        print(f"Macro -> Precision: {macro_precision:.4f} | Recall: {macro_recall:.4f} | F1-Score: {macro_f1:.4f}")
        print(f"Micro -> Precision: {micro_precision:.4f} | Recall: {micro_recall:.4f} | F1-Score: {micro_micro_f1:.4f}")
        
        # Save model weights based on validation loss
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_macro_f1 = macro_f1
            torch.save(model.state_dict(), f"best_distilbert_fold_{fold_id}.pt")
            print(f"--> Best model saved (Val Loss: {best_val_loss:.4f})")
            
    return best_macro_f1

In [14]:
target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

# Hem train hem test setindeki etiketleri int tipine zorluyoruz
for col in target_cols:
    train_df[col] = train_df[col].astype(int)
    test_df[col] = test_df[col].astype(int)

In [15]:
if __name__ == '__main__':
    # Device configuration for Apple Silicon (MPS), CUDA, or CPU
    if torch.backends.mps.is_available():
        device = torch.device("mps")
        print("Using Apple Silicon GPU (MPS) for training.")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
        print("Using Nvidia GPU (CUDA) for training.")
    else:
        device = torch.device("cpu")
        print("Using CPU for training. Note: This might be slow.")
        
    # Initialize the Fast Tokenizer
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    
    # List to store the best Macro-F1 score of each fold
    cv_macro_f1_scores = []

    # Execute 5-Fold Cross-Validation
    for fold in range(5):
        # The function now takes epochs=1 internally, but we state it explicitly for clarity
        best_fold_f1 = train_and_evaluate_fold(
            fold_id=fold, 
            train_df=train_df, 
            tokenizer=tokenizer, 
            device=device, 
            epochs=1, 
            batch_size=64
        )
        cv_macro_f1_scores.append(best_fold_f1)

    print("\n==================== 5-FOLD CV FINAL REPORT ====================")
    print(f"All Fold Macro-F1 Scores : {[round(score, 4) for score in cv_macro_f1_scores]}")
    print(f"Mean CV Macro-F1 Score   : {np.mean(cv_macro_f1_scores):.4f} (+/- {np.std(cv_macro_f1_scores):.4f})")
    print("================================================================")

Using Apple Silicon GPU (MPS) for training.

--- FOLD 0 TRAINING ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Epoch 1/1 ---
Batch 0/80 | Batch Loss: 0.7203
Batch 20/80 | Batch Loss: 0.3508
Batch 40/80 | Batch Loss: 0.3058
Batch 60/80 | Batch Loss: 0.1801

Evaluating on validation set...

--- Epoch 1 Summary Report ---
Train Loss: 0.3325 | Val Loss: 0.1932

Per-Class Performance:
Label           | Precision  | Recall     | F1-Score  
-------------------------------------------------------
toxic           | 0.8685     | 0.7822   | 0.8231
severe_toxic    | 0.0000     | 0.0000   | 0.0000
obscene         | 0.7957     | 0.7276   | 0.7602
threat          | 0.0000     | 0.0000   | 0.0000
insult          | 0.7117     | 0.6723   | 0.6915
identity_hate   | 0.0000     | 0.0000   | 0.0000

Global Averages:
Macro -> Precision: 0.3960 | Recall: 0.3637 | F1-Score: 0.3791
Micro -> Precision: 0.8097 | Recall: 0.6823 | F1-Score: 0.7405
--> Best model saved (Val Loss: 0.1932)

--- FOLD 1 TRAINING ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Epoch 1/1 ---
Batch 0/81 | Batch Loss: 0.6841
Batch 20/81 | Batch Loss: 0.3484
Batch 40/81 | Batch Loss: 0.2608
Batch 60/81 | Batch Loss: 0.2497
Batch 80/81 | Batch Loss: 0.0957

Evaluating on validation set...

--- Epoch 1 Summary Report ---
Train Loss: 0.2967 | Val Loss: 0.1744

Per-Class Performance:
Label           | Precision  | Recall     | F1-Score  
-------------------------------------------------------
toxic           | 0.8959     | 0.8371   | 0.8655
severe_toxic    | 0.0000     | 0.0000   | 0.0000
obscene         | 0.8118     | 0.8150   | 0.8134
threat          | 0.0000     | 0.0000   | 0.0000
insult          | 0.6894     | 0.7200   | 0.7043
identity_hate   | 0.0000     | 0.0000   | 0.0000

Global Averages:
Macro -> Precision: 0.3995 | Recall: 0.3953 | F1-Score: 0.3972
Micro -> Precision: 0.8184 | Recall: 0.7203 | F1-Score: 0.7662
--> Best model saved (Val Loss: 0.1744)

--- FOLD 4 TRAINING ---


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Epoch 1/1 ---
Batch 0/81 | Batch Loss: 0.6641
Batch 20/81 | Batch Loss: 0.3214
Batch 40/81 | Batch Loss: 0.2717
Batch 60/81 | Batch Loss: 0.2329
Batch 80/81 | Batch Loss: 0.2683

Evaluating on validation set...

--- Epoch 1 Summary Report ---
Train Loss: 0.3061 | Val Loss: 0.1842

Per-Class Performance:
Label           | Precision  | Recall     | F1-Score  
-------------------------------------------------------
toxic           | 0.8875     | 0.7904   | 0.8361
severe_toxic    | 0.0000     | 0.0000   | 0.0000
obscene         | 0.8041     | 0.7912   | 0.7976
threat          | 0.0000     | 0.0000   | 0.0000
insult          | 0.6808     | 0.6197   | 0.6488
identity_hate   | 0.0000     | 0.0000   | 0.0000

Global Averages:
Macro -> Precision: 0.3954 | Recall: 0.3669 | F1-Score: 0.3804
Micro -> Precision: 0.8115 | Recall: 0.6815 | F1-Score: 0.7409
--> Best model saved (Val Loss: 0.1842)

==================== 5-FOLD CV FINAL REPORT ====================
All Fold Macro-F1 Scores : [0.3791,

In [5]:
def evaluate_distilbert_ensemble(test_csv_path, model_dir=".", batch_size=64):
    print("\n--- STARTING DISTILBERT ENSEMBLE INFERENCE ON TEST SET ---")
    
    # 1. Cihaz ve Tokenizer Kurulumu
    device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
    tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
    target_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']
    
    # 2. Ortak 200 Satırlık Test Setini Yükleme
    test_df = pd.read_csv(test_csv_path)
    test_dataset = ToxicCommentDataset(test_df, tokenizer, max_len=256)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=False)
    
    # Tüm katların olasılık tahminlerini toplamak için boş bir matris hazırlıyoruz
    # Boyut: (Kat Sayısı, Test Satır Sayısı, Sınıf Sayısı) -> (5, 200, 6)
    all_fold_probs = []
    
    # Real ground truth etiketleri alalım
    true_labels = test_df[target_cols].values
    
    # 3. 5 Katın Modellerini Sırayla Yükleyip Tahmin Alma
    for fold_id in range(5):
        print(f"Loading weights for Fold {fold_id}...")
        
        # Boş model mimarisini ayağa kaldır ve cihazına (MPS) taşı
        model = DistilBertMultiLabelClassifier().to(device)
        
        # Kaydedilen ağırlıkları modele enjekte et
        model_path = f"{model_dir}/best_distilbert_fold_{fold_id}.pt"
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        fold_probs = []
        with torch.no_grad():
            for batch in test_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                
                logits = model(input_ids, attention_mask)
                # Sınıflandırma olasılıklarını hesapla (0 ile 1 arası)
                probs = torch.sigmoid(logits).cpu().numpy()
                fold_probs.append(probs)
                
        fold_probs = np.vstack(fold_probs)
        all_fold_probs.append(fold_probs)
        
    # 4. ENSEMBLE ADIMI: 5 modelin verdiği olasılıkların aritmetik ortalamasını alıyoruz
    # Bu hamle varyansı düşürür ve modeli çok daha kararlı hale getirir
    ensemble_probs = np.mean(all_fold_probs, axis=0)
    
    # Eşik değerine göre ikili matrise çevirme (Threshold = 0.5)
    binary_preds = (ensemble_probs >= 0.5).astype(int)
    
    # --- METRİK HESAPLAMALARI ---
    macro_precision = precision_score(true_labels, binary_preds, average='macro', zero_division=0)
    macro_recall = recall_score(true_labels, binary_preds, average='macro', zero_division=0)
    macro_f1 = f1_score(true_labels, binary_preds, average='macro', zero_division=0)
    
    micro_precision = precision_score(true_labels, binary_preds, average='micro', zero_division=0)
    micro_recall = recall_score(true_labels, binary_preds, average='micro', zero_division=0)
    micro_f1 = f1_score(true_labels, binary_preds, average='micro', zero_division=0)
    
    print("\n==================== DISTILBERT ENSEMBLE TEST REPORT ====================")
    print(f"Macro -> Precision: {macro_precision:.4f} | Recall: {macro_recall:.4f} | F1-Score: {macro_f1:.4f}")
    print(f"Micro -> Precision: {micro_precision:.4f} | Recall: {micro_recall:.4f} | F1-Score: {micro_f1:.4f}")
    print("=========================================================================")

if __name__ == '__main__':
    # Ortak test dosyanın adını buraya yazıyorsun
    evaluate_distilbert_ensemble(test_csv_path="mini_test_sample.csv")


--- STARTING DISTILBERT ENSEMBLE INFERENCE ON TEST SET ---
Loading weights for Fold 0...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights for Fold 1...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights for Fold 4...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



==================== DISTILBERT ENSEMBLE TEST REPORT ====================
Macro -> Precision: 0.3754 | Recall: 0.3628 | F1-Score: 0.3685
Micro -> Precision: 0.7872 | Recall: 0.6789 | F1-Score: 0.7291
